<a href="https://www.kaggle.com/code/oladigbolutaofeek/social-media-ads-notebook-ipynb?scriptVersionId=303336755" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# 📣 Social Media Advertising Analytics
## A Complete EDA of Channel Performance, Audience Behaviour & Campaign Efficiency

---

> *"Half the money I spend on advertising is wasted — the trouble is, I don't know which half."*
> — John Wanamaker, 1919

**A century later, we finally have the data to answer him.**

---

## 🧭 What This Notebook Is About

Every business running social media ads faces the same three questions:

- 📍 **Where** should I put my budget — which channel actually delivers?
- 👥 **Who** should I target — which audience converts at lowest cost?
- ⏰ **When** should I run campaigns — are there peak windows worth exploiting?

This notebook works through a social media advertising dataset to answer all three — using structured EDA, advanced visualisations, and statistical testing to separate signal from noise.

---

## 📦 Dataset Overview

| Feature | Description |
|---|---|
| `Campaign_ID` | Unique identifier per campaign |
| `Channel_Used` | Platform — Instagram, YouTube, Facebook, Twitter, Pinterest |
| `Campaign_Goal` | Objective — Awareness, Conversion, Retention, etc. |
| `Target_Audience` | Audience demographic segment |
| `Customer_Segment` | Customer classification |
| `ROI` | Return on investment |
| `Conversion_Rate` | % of impressions that converted |
| `Acquisition_Cost` | Cost per acquired customer |
| `Clicks` / `Impressions` | Raw engagement volume |
| `Engagement_Score` | Platform-level engagement rating |
| `Duration` | Campaign length in days |
| `Location` / `Language` | Geographic & language targeting |

**Derived KPIs we engineer:**
- `CTR` = Clicks ÷ Impressions
- `Cost_per_Click` = Acquisition_Cost ÷ Clicks
- `ROI_per_Day` = ROI ÷ Duration
---

## 🗺️ Notebook Roadmap

| # | Section | Key Question |
|---|---------|-------------|
| 1 | Setup & Loading | What does the data look like? |
| 2 | Cleaning & Feature Engineering | What KPIs are we missing? |
| 3 | KPI Distributions | Are the numbers healthy or skewed? |
| 4 | Channel Showdown 🥊 | Which platform delivers the best ROI? |
| 5 | Campaign Goals | Does goal type predict performance? |
| 6 | Audience Intelligence | Who clicks? Who actually converts? |
| 7 | Efficiency Quadrant | High ROI at low cost — who's in the sweet spot? |
| 8 | Does Engagement Convert? | Is a high engagement score actually meaningful? |
| 9 | Summary | What actually work? |

---

## 🛠️ Tech Stack

```python
pandas      # data wrangling
numpy       # numerical operations
matplotlib  # charting engine
seaborn     # statistical visualisations
```

---

## 👥 Who This Is For

| Level | What you'll get |
|---|---|
| 🟢 **Beginners** | Every code block is explained in plain English |
| 🟡 **Intermediate** | Reusable EDA patterns and chart templates |
| 🔴 **Advanced** | Statistical validation approach and derived KPI design |

---

> 📌 *If this notebook helps you, an upvote keeps it visible to others — it's the best way to support the work. Feel free to fork, adapt, and build on it.*

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pylab as plt
import seaborn as sns
plt.style.use('ggplot')
#pd.set_option('max_columns', 200)
plt.rcParams.update({
    "axes.labelsize": 48,   # x and y axis titles
    "axes.titlesize": 30,   # plot title
    "xtick.labelsize": 36,  # x tick labels
    "ytick.labelsize": 36   # y tick labels
})

# Data Loading 
~ bring raw data into memory

In [ ]:
df = pd.read_excel('/kaggle/input/datasets/oladigbolutaofeek/social-media-advertising-analysis/Social_Media_Advertising.xlsx')

# Data Understanding & Inspection
~ Purpose: understand the overall                     structure of the data  

In [ ]:
df.shape  # (rows, columns)

In [ ]:
df.head() # first 5 rows

In [ ]:
df.tail() # last 5 rows

In [ ]:
df.columns  # column names

In [ ]:
df.dtypes  # column-by-column type summary

In [ ]:
df.describe()  # numeric columns

# Data Preparation
~ purpose: identify missing values, 
           format & clean data

### Missing data identification 
~ purpose: identify gaps in the data

In [ ]:
# Check for missing values
df.isna().sum() 

## Data cleaning
~ purpose: fix or remove problems found in the data

In [ ]:
 # Check for duplicate columns
df.loc[df.duplicated()]

In [ ]:
# Drop duplicate rows
df= df.drop_duplicates()  

In [ ]:
# Change date format
df["Date"] = pd.to_datetime(df["Date"])

In [ ]:
# Change duration from datatype object to float
df['Duration_Days']= df['Duration'].str.extract(r'(\d+\.?\d*)').astype(float)

## Feature Engineering
~ purpose: derive new variables that may              carry predictive signal not                available in the data

In [ ]:
# Extract temporal features from the timestamp
df["hour"]      = df["Date"].dt.hour
df["day"] = df["Date"].dt.day_name()
df["month"]     = df["Date"].dt.month

In [ ]:
# Derive metrics (CTR)
df['CTR']= df['Clicks']/df['Impressions'].replace(0, np.nan)  # Normalizes click volume

In [ ]:
# Derive metrics (CPC)
df['Cost_per_Click']= df['Acquisition_Cost']/df['Clicks'].replace(0, np.nan)  # Efficiency metric

In [ ]:
# Derive metrics (ROIPD)
df['ROI_per_Day']= df['ROI']/df['Duration_Days'].replace(0, np.nan)  # Controls for campaign length

# Exploratory Data Analysis (EDA)
~ Purpose: Uncover distributions,                     relationships, and patterns                that guide modelling decisions.

## Univariate Analysis: Numeric Distribution
~ Purpose: understand numeric data

In [ ]:
numeric_cols = [
    "ROI", "Conversion_Rate", "Acquisition_Cost",
    "Clicks", "Impressions", "Engagement_Score",
    "Duration", "CTR", "Cost_per_Click", "ROI_per_Day"
]

fig, axes = plt.subplots(2, 5, figsize=(150, 75))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    axes[i].hist(df[col].dropna(), bins=30, color="steelblue", edgecolor="white")
    axes[i].set_title(col, fontsize=75)
    axes[i].set_xlabel("")
    axes[i].set_ylabel("Count")

plt.suptitle("Numeric Feature Distributions", fontsize=150, y=1.01)
plt.show()

## Univariate Analysis: Categorical Distribution
~ Purpose: understand categorical data

In [ ]:
cat_cols = [
    "Channel_Used", "Campaign_Goal", "Target_Audience",
    "Customer_Segment", "Location", "Language", "Company"
]
fig, axes= plt.subplots(2, 4, figsize= (150, 70))
axes= axes.flatten()
for i, col in enumerate(cat_cols):
    order= df[col].value_counts().index
    sns.countplot(data= df, y= col, order= order, ax= axes[i], palette= "Blues_r", hue= col, legend= None)
    axes[i].set_title(f'{col}_Count', fontsize= 75)
    axes[i].set_xlabel('Count')
    axes[i].set_ylabel('')
axes[-1].set_visible(False)
plt.suptitle('Categorical Feature Distribution', fontsize= 150, y=1.01)
plt.show()


## Bivariate Analysis
~ Purpose: identify what drives performance

#### Correlation Matrix
~ Purpose: under relationship between metrics

In [ ]:
corr_cols = [
    "ROI", "Conversion_Rate", "Acquisition_Cost",
    "Clicks", "Impressions", "Engagement_Score",
    "Duration_Days", "CTR", "Cost_per_Click", "ROI_per_Day"
]

plt.figure(figsize=(13, 10))
corr= df[corr_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
ax = sns.heatmap(
    corr, mask=mask, annot=True, fmt=".2f",
    cmap="coolwarm", center=0, square=True,
    linewidths=0.5, cbar_kws={"shrink": 0.8},
    annot_kws={"size": 11}
)
ax.set_xticklabels(ax.get_xticklabels(), fontsize=9, rotation=90, ha="right")
ax.set_yticklabels(ax.get_yticklabels(), fontsize=9, rotation=0)
ax.collections[0].colorbar.ax.tick_params(labelsize=9)
plt.title("Correlation Matrix — Numeric KPIs", fontsize=14)
plt.tight_layout()
plt.show()

## Channel Performance
~ Purpose: understand how effective ads channels

In [ ]:
channel_kpis = (
    df.groupby("Channel_Used")[["ROI", "Conversion_Rate", "CTR", "Acquisition_Cost"]]
    .mean()
    .round(3)
    .sort_values("ROI", ascending=False)
)
print("\nChannel KPI Summary:\n", channel_kpis)
 
fig, axes = plt.subplots(1, 4, figsize=(90, 20))
kpis = ["ROI", "Conversion_Rate", "CTR", "Acquisition_Cost"]
colors = ["steelblue", "seagreen", "darkorange", "tomato"]
 
for ax, kpi, color in zip(axes, kpis, colors):
    data = channel_kpis[kpi].sort_values(ascending=True)
    ax.barh(data.index, data.values, color=color, edgecolor="white")
    ax.set_title(f"Mean {kpi} by Channel", fontsize=45)
    ax.set_xlabel(kpi)
 
plt.suptitle("Channel Performance Comparison", fontsize=70)
plt.show()
 

## Target Audience & Customer Segment
~ Purpose: understand demographics of audience

In [ ]:
for group_col in ["Target_Audience", "Customer_Segment"]:
    grp= (
        df.groupby(group_col)[["ROI", "Conversion_Rate", "Engagement_Score", "Acquisition_Cost"]]
        .mean()
        .round(3)
        .sort_values('ROI', ascending= False)
    )
print(f"\n{group_col} KPI Summary:\n", grp)

fig, axes = plt.subplots(1, 4, figsize=(90, 20))
for ax, kpi, color in zip(axes,
                              ["ROI", "Conversion_Rate", "Engagement_Score", "Acquisition_Cost"],
                              ["steelblue", "seagreen", "darkorange", "tomato"]):
       data = grp[kpi].sort_values(ascending=True)
       ax.barh(data.index, data.values, color=color, edgecolor="white")
       ax.set_title(f"Mean {kpi} by {group_col}", fontsize=45)
plt.suptitle(f"{group_col} Performance", fontsize=70)
plt.show()
 

## Duration vs ROI
~ Purpose: influence of ads duration on return on investment 

In [ ]:
plt.figure(figsize= (10, 5))
ax= sns.scatterplot(data= df, x= 'Duration', y= 'ROI', 
                hue= 'Duration', size= 'ROI', sizes= (50, 300), 
                alpha=0.7, legend= 'brief'
               )
ax.set_xlabel("Duration", fontsize=10)
ax.set_ylabel("ROI", fontsize=10)
ax.tick_params(axis="both", labelsize=9) 
ax.legend(fontsize=9, title_fontsize=10)
plt.title('Duration vs ROI', fontsize= 14)
plt.show()

## Acquisition Cost vs ROI  (Efficiency Quadrant)
~ Purpose:

In [ ]:
plt.figure(figsize=(10, 5))
sns.scatterplot(data=df, x="Acquisition_Cost", y="ROI",
                hue="Channel_Used", size="Engagement_Score",
                sizes=(30, 200), alpha=0.7)
plt.axhline(df["ROI"].median(), color="gray", linestyle="--", linewidth=1, label="Median ROI")
plt.axvline(df["Acquisition_Cost"].median(), color="gray", linestyle=":", linewidth=1,
            label="Median Cost")
plt.title("Acquisition Cost vs ROI — Efficiency Quadrant", fontsize= 14)
plt.xlabel('Acquisition Cost', fontsize= 10)
plt.ylabel('ROI', fontsize= 10)
plt.xticks(fontsize=9)
plt.yticks(fontsize=9)
plt.legend(bbox_to_anchor=(1.01, 1))
plt.show()

## Channel × Campaign Goal

In [ ]:
for metric in ["ROI", "Conversion_Rate", "CTR"]:
    pivot = df.pivot_table(values=metric, index="Campaign_Goal",
                           columns="Channel_Used", aggfunc="mean")
    plt.figure(figsize=(12, 5))
    ax= sns.heatmap(pivot, annot=True, fmt=".3f", cmap="YlGnBu", linewidths=1)
    ax.set_xlabel(ax.get_xlabel(), fontsize=12)
    ax.set_ylabel(ax.get_xlabel(), fontsize=12)
    ax.set_xticklabels(ax.get_xticklabels(), fontsize= 9)
    ax.set_yticklabels(ax.get_yticklabels(), fontsize= 9)
    ax.collections[0].colorbar.ax.tick_params(labelsize=9)
    plt.title(f"Mean {metric} — Channel × Campaign Goal", fontsize= 14)
    plt.show()
 

### Multi-dimensional Analysis
~ Purpose: identify relationships between variables

## Top Company Performance
~ Purpose: identify best performing company

In [ ]:
company_perf = (
    df.groupby("Company")[["ROI", "Conversion_Rate", "CTR", "Acquisition_Cost"]]
    .mean()
    .round(3)
    .sort_values("ROI", ascending=False)
    .head(15)
)
print("\nTop 15 Companies by ROI:\n", company_perf)

# Summary Table
── Best performing combination of Channel × Goal

In [ ]:
summary = (
    df.groupby(["Channel_Used", "Campaign_Goal"])
    .agg(
        Mean_ROI=("ROI", "mean"),
        Mean_Conversion=("Conversion_Rate", "mean"),
        Mean_CTR=("CTR", "mean"),
        Mean_Cost=("Acquisition_Cost", "mean"),
        Campaigns=("Campaign_ID", "count")
    )
    .round(3)
    .sort_values("Mean_ROI", ascending=False)
    .reset_index()
)
sns.tableplot()
 
print("\nTop Channel × Goal Combinations by ROI:")
print(summary.head(10).to_string(index=False))